
# Gold Customer Metrics

Creates customer-level analytical metrics for the GenAI Data Analyst Copilot.

Target:

`genai_copilot.gold.customer_metrics`

In [0]:
from pyspark.sql import functions as F

In [0]:
SILVER_TABLE = "genai_copilot.silver.sales"
CUSTOMER_TABLE = "genai_copilot.gold.customer_metrics"

silver_df = spark.table(SILVER_TABLE)

print("Silver rows:", silver_df.count())

In [0]:
customer_metrics = (
    silver_df
    .groupBy(
        "customer_id",
        "customer_name",
        "country"
    )
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_quantity"),
        F.sum("revenue").alias("total_revenue"),
        F.sum("cost").alias("total_cost"),
        F.sum("profit").alias("total_profit"),
        F.avg("revenue").alias("average_order_value"),
        F.min("order_date").alias("first_order_date"),
        F.max("order_date").alias("last_order_date")
    )
    .withColumn(
        "profit_margin",
        F.when(
            F.col("total_revenue") > 0,
            F.col("total_profit") / F.col("total_revenue")
        ).otherwise(F.lit(0.0))
    )
    .orderBy(
        F.desc("total_revenue")
    )
)

In [0]:
display(customer_metrics.limit(20))

In [0]:
(
    customer_metrics
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(CUSTOMER_TABLE)
)

print(f"Created {CUSTOMER_TABLE}")

In [0]:
%sql
SELECT
    customer_id,
    customer_name,
    total_orders,
    total_revenue,
    total_profit,
    profit_margin
FROM genai_copilot.gold.customer_metrics
ORDER BY total_revenue DESC
LIMIT 10;

In [0]:
%sql
SELECT
    ROUND(SUM(revenue), 2) AS silver_revenue
FROM genai_copilot.silver.sales;

In [0]:
%sql
SELECT
    ROUND(SUM(total_revenue), 2) AS gold_revenue
FROM genai_copilot.gold.region_sales;

In [0]:
%sql
SELECT
    ROUND(SUM(profit), 2) AS silver_profit
FROM genai_copilot.silver.sales;

In [0]:
%sql
SELECT
    ROUND(SUM(total_profit), 2) AS gold_profit
FROM genai_copilot.gold.region_sales;

In [0]:
%sql
SHOW TABLES IN genai_copilot.gold;